In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# Cargar el dataset
data = pd.read_csv('penguins_size.csv')
data = data[data.sex != '.']
data.dropna(inplace=True)
data.reset_index(drop=True, inplace=True)
data.head()

**Relación entre variables**

In [ ]:
sns.pairplot(data)

## 2. Entrenamiento de modelos de regresión

**Regresión lineal con un atributo - 1D**

### Estimando coeficientes

In [ ]:
np.random.seed(50)
n = 20
x = np.linspace(-2, 2, n)
a = 2
b = 3
sigma = 0.75
y = a*x+b + sigma*np.random.randn(n)

In [ ]:
plt.scatter(x, y)
plt.plot(x, a*x+b, 'r', label = f'y = {a}x + {b}')
plt.legend()

In [ ]:
from ipywidgets import interactive

# Define the cost function
def rss(a, b, x=x, y=y):
    # return  np.sum(np.abs(y - a * x - b))
    return np.sum((y - a * x - b) ** 2)

# Define the function to update the plot
def update_plot(a, b):
    plt.figure(figsize=(6, 4))
    plt.scatter(x, y, label='Data points')
    y_pred = a * x + b
    plt.plot(x, y_pred, color='red', label=f'Line: y = {a:.2f}x + {b:.2f}')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True)

    # Display the cost
    plt.text(1, y.min(), f'RSS: {rss(a, b):.2f}', fontsize=12, bbox=dict(facecolor='yellow', alpha=0.5))

    plt.show()

# Create interactive widgets
interactive_plot = interactive(update_plot, a=(-10.0, 10.0, 0.1), b=(-10, 10, 0.1))
interactive_plot

In [ ]:
# Generate a grid of a and b values
a_ = 2
b_range = np.linspace(-8, 8, 100)
C = np.zeros_like(b_range)

for j, b_ in enumerate(b_range):
    C[j] = rss(a_, b_)

# Create the 3D plot of the cost surface
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)
ax.plot(b_range, C)
ax.set_title('Cost function (1 parameter)')
ax.set_xlabel('b (intercept)')
ax.set_ylabel('Cost')

plt.show()

In [ ]:
# Generate a grid of a and b values
a_range = np.linspace(-8, 8, 100)
b_range = np.linspace(-8, 8, 100)
A, B = np.meshgrid(a_range, b_range)
C = np.zeros_like(A)

# Calculate cost for each pair of a and b
for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        C[i, j] = rss(A[i, j], B[i, j])

# Create the 3D plot of the cost surface
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(A, B, C, cmap='viridis', edgecolor='none')
ax.set_title('Cost Surface')
ax.set_xlabel('a (slope)')
ax.set_ylabel('b (intercept)')
ax.set_zlabel('Cost')
fig.colorbar(surf, shrink=0.5, aspect=5)

# Point to add the vertical line at
cost_point = rss(a, b)

# Add the vertical line
ax.plot([a, a], [b, b], [0, cost_point], marker='o', color='red')

plt.show()

In [ ]:
### Countour plot of the cost function

plt.figure(figsize=(8, 6))
contour = plt.contour(A, B, C, levels=20)
plt.clabel(contour, inline=True, fontsize=8)
plt.xlabel('a (slope)')
plt.ylabel('b (intercept)')
plt.scatter(a, b, color='red', label='Punto óptimo')
plt.axvline(a, color='red', linestyle='--')
plt.axhline(b, color='red', linestyle='--')

Volvamos al dataset de pingüinos

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error

In [ ]:
# Separar características y etiqueta
X = data[['flipper_length_mm']]
y = data['body_mass_g']

In [ ]:
# Dividir el dataset en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

**Scipy**

In [ ]:
from scipy import stats

# Crear el modelo de regresión lineal
res = stats.linregress(X_train['flipper_length_mm'], y_train)
print(res)


In [ ]:
print(f"Intercept: {res.intercept:.2f}, Slope: {res.slope:.2f}")
print(f"R-squared: {res.rvalue**2:.2f}")
print(f"slope (95%): {res.slope:.1f} +/- {1.96*res.stderr:.1f}") # Aproximación
print(f"intercept (95%): {res.intercept:.0f} +/- {1.96*res.intercept_stderr:.0f}")


**Visualización**

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(6, 4))

axs.scatter(X_train.flipper_length_mm, y_train)
axs.set_xlabel('Longitud de la aleta (mm)')
axs.set_ylabel('Masa corporal (g)')

x_plot = np.linspace(160, 240, 100)

axs.plot(x_plot, res.intercept + res.slope*x_plot, color='red', label = 'SciPy')
axs.legend()


**Residuos**

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(6, 4))

axs.scatter(X_train.flipper_length_mm, y_train)
axs.set_xlabel('Longitud de la aleta (mm)')
axs.set_ylabel('Masa corporal (g)')

x_plot = np.linspace(160, 240, 100)

for x_point, y_point in zip(X_train.flipper_length_mm, y_train):
    y_predicted = res.intercept + res.slope*x_point
    axs.plot([x_point, x_point], [y_point, y_predicted], color='k')

axs.plot(x_plot, res.intercept + res.slope*x_plot, color='red', label='SciPy')
axs.legend()



**Scikit-learn**

In [ ]:
reg = LinearRegression()
reg.fit(X_train, y_train)

In [ ]:
print('Pendiente: ', reg.coef_) # Notar que es un arreglo, ¿por qué?
print('Intersección: ', reg.intercept_)

**Visualización**

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(6, 4))

axs.scatter(X_train.flipper_length_mm, y_train)
axs.set_xlabel('Longitud de la aleta (mm)')
axs.set_ylabel('Masa corporal (g)')

x_plot = np.linspace(160, 240, 100)

axs.plot(x_plot, res.intercept + res.slope*x_plot, color='red', label='SciPy')
axs.plot(x_plot, reg.intercept_ + reg.coef_[0]*x_plot, color='green', label='Scikit-learn')

axs.legend()

### Evaluación del modelo

#### Métricas

In [ ]:
# Predecir y evaluar el modelo
y_test_pred = reg.predict(X_test)

print(f'Raiz del error cuadrático medio: {root_mean_squared_error(y_test, y_test_pred):.2f}')
print(f'Coeficiente de determinación: {r2_score(y_test, y_test_pred):.2f}')

#### Gráficos

**Predicciones vs. Valores reales**

In [ ]:
plt.scatter(y_test, y_test_pred, alpha=0.75)
plt.plot([2700,6000], [2700,6000], ls='--', c='r', label='identidad: y = x')
plt.xlabel('y')
plt.ylabel('y_pred')
plt.legend()
plt.show()

**Distribución de los residuos**

In [ ]:
res = y_test_pred - y_test
plt.hist(res, bins = 20, rwidth = 0.9)
plt.xlabel('res')
plt.show()

Preguntas:
1. ¿Qué ocurre cuando hay más de un atributo predictor?
1. ¿Y si alguno de esos atributos es categórico?

## Regresión lineal múltiple

In [ ]:
### matriz de correlaciones

sns.heatmap(data[['flipper_length_mm', 'culmen_length_mm', 'culmen_depth_mm','body_mass_g']].corr(),
            annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

In [ ]:
# Separar características y etiqueta
X = data[['flipper_length_mm','culmen_length_mm', 'culmen_depth_mm']]
y = data['body_mass_g']

In [ ]:
# Dividir el dataset en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [ ]:
reg = LinearRegression()
reg.fit(X_train, y_train)

In [ ]:
print('Pendiente: ', reg.coef_)
print('Intersección: ', reg.intercept_)

In [ ]:
# Predecir y evaluar el modelo
y_test_pred = reg.predict(X_test)

print(f'Error cuadrático medio: {root_mean_squared_error(y_test, y_test_pred):.2f}')
print(f'Coeficiente de determinación: {r2_score(y_test, y_test_pred):.2f}')

#### Gráficos

**Predicciones vs. Valores reales**

In [ ]:
plt.scatter(y_test,y_test_pred, alpha=0.75)
plt.plot([2700,6000],[2700,6000], ls='--', c='r', label='identidad')
plt.xlabel('y')
plt.ylabel('y_pred')
plt.legend()
plt.show()

**Distribución de los residuos**

In [ ]:
res = y_test_pred - y_test
plt.hist(res, bins = 20, rwidth = 0.9)
plt.xlabel('res')
plt.show()

### Atributos categóricos

In [ ]:
sns.scatterplot(data=data, x='culmen_length_mm', y='culmen_depth_mm', hue='species')
X = data[['culmen_length_mm']]
y = data.culmen_depth_mm

reg = LinearRegression()
reg.fit(X, y)

x_plot = np.linspace(30, 60, 100).reshape(-1, 1)
y_plot = reg.predict(x_plot)
plt.plot(x_plot, y_plot, color='red')

¿Y si agregamos las variables categóricas?

In [ ]:
data['dummy_adelie'] = (data['species'] == 'Adelie').astype(int)
data['dummy_chinstrap'] = (data['species'] == 'Chinstrap').astype(int)

In [ ]:
# Separar características y etiqueta
X = data[['culmen_length_mm', 'dummy_adelie', 'dummy_chinstrap']]
y = data['culmen_depth_mm']

In [ ]:
# Dividir el dataset en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [ ]:
reg = LinearRegression()
reg.fit(X_train, y_train)

In [ ]:
print('Pendientes: ', reg.coef_)
print('Intersección: ', reg.intercept_)

**Visualización**

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(6, 4))
x_plot = np.linspace(28, 60, 100)

for species in data['species'].unique():

    if species == 'Adelie':
        mask = X_train['dummy_adelie'].astype(bool)
        axs.scatter(X_train[mask]['culmen_length_mm'], y_train[mask], label='Adelie')
        axs.plot(x_plot, reg.intercept_ + reg.coef_[1] +reg.coef_[0]*x_plot)

    if species == 'Chinstrap':
        mask = X_train['dummy_chinstrap'].astype(bool)
        axs.scatter(X_train[mask]['culmen_length_mm'], y_train[mask], label='Chinstrap')
        axs.plot(x_plot, reg.intercept_ + reg.coef_[2] +reg.coef_[0]*x_plot)

    else:
        mask = np.logical_or(X_train['dummy_chinstrap'], X_train['dummy_adelie'])
        mask = ~mask
        axs.scatter(X_train[mask]['culmen_length_mm'], y_train[mask], label='Gentoo')
        axs.plot(x_plot, reg.intercept_ + reg.coef_[0]*x_plot)

axs.set_xlabel('culmen_length_mm')
axs.set_ylabel('culmen_depth_mm')

axs.legend()

In [ ]:
reg.fit(X_train, y_train)
y_test_pred = reg.predict(X_test)
plt.scatter(y_test,y_test_pred, alpha=0.75)
plt.plot([12,22],[12,22], ls='--', c='r', label='identidad: y = x')
plt.xlabel('y')
plt.ylabel('y_pred')
plt.legend()
plt.show()